# 23. Speculative Decoding | 投机解码
**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `解码`, `Speculative Decoding` | **目标人群：** 推理优化学习者

---

## 本节导读

自回归生成中，目标模型通常要逐个 token 推进；当每一步都要重复进入模型时，单步成本会直接累积到整段输出。本节从已有的 KV Cache 状态视角出发，引入草稿模型：先提出多个候选，再由目标模型集中验证。

学习路径沿着“提议 → 验证 → 接受或修正”展开：先看候选如何与目标概率对齐，再理解接受概率、residual correction 和 bonus token 分别解决什么问题。最后把这些状态变化串成一轮生成，理解减少逐 token 推进的条件。

**关键词：** `speculative decoding`, `draft model`, `verification`

---

## 前置阅读

**导语：** 先掌握单步解码和 KV Cache 的状态变化，再理解草稿模型如何提出候选、目标模型如何批量验证。
- [21. Decoding Strategies | 解码策略](./21_Decoding_Strategies.ipynb)
- [P1: 11. KV Cache and Memory Growth | KV Cache 与显存增长](../01_Hardware_Math_and_Systems/11_KV_Cache_and_Memory_Growth.ipynb)

---


### Step 1: 投机式生成策略如何减少目标模型推进

投机解码不是单一实现，而是一类“先并行提出候选，再由更可靠路径验证”的生成加速策略。本节聚焦最经典的 draft / target 模型协作；多 Token、多候选树或模型内部草稿头都共享“提高每轮有效推进量”的目标，但验证接口与质量约束不同。

| 策略分支 | 候选从哪里来 | 本节 / 后续入口 | 共同判断标准 |
|---|---|---|---|
| draft / target | 小草稿模型提出连续 token | 本节核心 | 接受率、验证成本、有效推进量 |
| multi-token / 多头候选 | 多个 token 或预测头并行提出候选 | [35](./35_Multi_Token_Decoding.ipynb) | 首次拒绝后的有效推进 |
| self-speculative / early exit | 同一模型的浅层或中间层提出候选 | 扩展概念，真实 backend 验证 | 额外草稿成本是否低于收益 |
| tree / multi-candidate | 同轮提出多条候选路径 | 扩展概念，关注验证结构 | 候选覆盖、tree verify 成本 |

一轮投机解码的输入是当前上下文和两个模型的候选分布，草稿模型（draft model）先提出一段 token，目标模型（target model）再集中验证。输出是已接受的 token、一个修正 token，或在全部通过时追加的 bonus token；两者的协作目标是减少目标模型逐 token 推进的次数，同时保持目标分布的生成语义。

| 参与者 | 负责什么 | 产生什么 |
|---|---|---|
| 草稿模型 | 快速提出连续候选 | `draft_tokens` 与 `draft_probs` |
| 目标模型 | 在对应位置验证候选 | `target_probs`，包含验证位置和 bonus 位置 |
| 单轮流程 | 根据验证结果决定推进长度 | 接受的 token、修正 token 或 bonus token |

![Speculative Decoding 流程图](../docs/public/02_PyTorch_Algorithms/23_speculative_decoding_flow.svg)

### Step 2: 草稿候选与目标概率如何对齐

一轮验证需要把草稿 token、草稿概率和目标概率按位置对齐。`K` 表示草稿 token 数，目标模型提供前 `K` 行来验证候选，并额外提供一行用于全部接受时的 bonus token。位置对齐是后续计算接受概率的前提。

| 输入 / 状态 | 形状或内容 | 用途 |
|---|---|---|
| `draft_probs` | `[K, vocab]` | 草稿模型在 K 个位置上的概率分布 |
| `target_probs` | `[K+1, vocab]` | 前 K 行验证草稿，最后一行提供 bonus token |
| `draft_tokens` | 长度为 K 的 token 序列 | 指出每个位置实际提出的候选 |
| 输入检查 | 非负、行归一化、长度和词表维度 | 防止概率与位置错位后继续计算 |

### Step 3: 接受、修正与 bonus 如何决定推进长度
按草稿位置从前到后处理：接受就继续验证，拒绝就从 residual distribution 采样修正 token 并结束本轮；全部 `K` 个草稿都接受时，再从目标模型的 bonus 位置采样一个 token。接受概率决定本轮能推进多长，公式为：

$$\alpha(x) = \min\left(1, \frac{p(x)}{q(x)}\right)$$

题目区验证经典 draft / target 路径的概率归一化、接受/拒绝、residual correction、bonus token 和控制流。不同策略分支的真实性能都取决于候选质量和验证结构；68 用统一 workload 检查 acceptance、目标模型 forward 次数、TTFT、TPOT 与吞吐。

| 结果 | 接受规则或使用的分布 | 本轮推进 |
|---|---|---|
| 接受 | `p(x) >= q(x)` 时必然接受；否则按 `p(x) / q(x)` 接受 | 进入下一个验证位置 |
| 拒绝 | 对 `max(target - draft, 0)` 归一化后采样 residual correction | 输出修正 token，结束本轮 |
| 全部接受 | 所有草稿 token 通过验证，使用最后一行 `target_probs` | 追加 bonus token |

![Speculative Decoding：验证结果决定下一步](../docs/public/02_PyTorch_Algorithms/23_speculative_acceptance_flow.svg)

### Step 4: 实现并验证单轮投机解码
本 Step 将前面的流程实现为 `speculative_decode_step`。输入是草稿 token、草稿概率和目标概率，输出至少包含最终 token、接受数量、是否拒绝和检查到的 target 位置。题目区按输入检查、逐位置接受、拒绝修正和全接受 bonus 四个阶段完成实现。

| 实现阶段 | 主要任务 | 验证重点 |
|---|---|---|
| 输入检查 | 检查概率形状、归一化和 token 位置 | 非法输入明确报错 |
| 接受判断 | 按位置计算接受概率并采样 | 接受数量和停止位置正确 |
| 修正与 bonus | 分别处理拒绝和全接受 | residual 与 bonus 分支不混淆 |

In [ ]:
import torch

In [ ]:
def _validate_probability_contract(draft_probs, target_probs, draft_tokens):
    """检查草稿、目标和候选 token 是否能按位置一一对应。"""
    # ==========================================
    # TODO 1：验证概率输入契约
    # 提示：先检查二维形状与 K / K+1 关系，再检查有限、非负、按行归一化；
    #       最后确认 draft_tokens 长度为 K，且每个 token_id 位于词表范围。
    # ==========================================
    # return K, vocab_size
    raise NotImplementedError('TODO 1：请验证概率输入契约')

def _accept_probability(target_prob, draft_prob):
    """返回当前候选的接受概率 alpha。"""
    # ==========================================
    # TODO 2：计算 alpha = min(1, p / q)
    # 变量：p = target_prob，q = draft_prob。q 为 0 时不要直接相除：
    #       p > 0 返回 1；p = 0 返回 0。
    # ==========================================
    # alpha = ???
    raise NotImplementedError('TODO 2：请计算接受概率')

def _normalized_residual(target_row, draft_row):
    """构造拒绝时用于 correction 的归一化 residual 分布。"""
    # ==========================================
    # TODO 3：构造拒绝位置的 correction 分布
    # 输入：target_row = target_probs[i]、draft_row = draft_probs[i]，两者形状均为 [vocab]。
    # 输出：residual 的形状仍为 [vocab]，元素非负且总和为 1，可直接传给 torch.multinomial。
    # 步骤：先计算 delta = target_row - draft_row；用 clamp(delta, min=0) 去掉负质量；
    #       再计算 residual_mass。若 residual_mass 为 0，应报错；否则返回 residual / residual_mass。
    # ==========================================
    # delta = ???
    # residual = ???
    # residual_mass = ???
    # return ???
    raise NotImplementedError('TODO 3：请构造 residual 分布')

def speculative_decode_step(draft_probs, target_probs, draft_tokens, generator=None):
    """
    验证 K 个草稿，并在拒绝或全接受时生成修正 token。
    
    Args:
        draft_probs: 草稿模型在 K 个位置上的概率分布, shape [K, vocab_size]
        target_probs: 目标模型在 K 个验证位置及 1 个 bonus 位置的概率分布, shape [K+1, vocab_size]
        draft_tokens: 草稿模型实际提出的 K 个 token_id, shape [K]
        
    Returns:
        dict: 包含最终 token 序列、接受数量、拒绝状态和检查位置。
    """
    K, vocab_size = _validate_probability_contract(draft_probs, target_probs, draft_tokens)
    accepted_tokens = []
    
    for i in range(K):
        token_id = draft_tokens[i]
        
        if not 0 <= int(token_id) < vocab_size:
            raise ValueError('draft token id 超出词表范围')
        p = target_probs[i, token_id]
        q = draft_probs[i, token_id]
        
        alpha = _accept_probability(p, q)
        if torch.rand((), generator=generator).item() < alpha:
            accepted_tokens.append(int(token_id))
            continue
        residual = _normalized_residual(target_probs[i], draft_probs[i])
        correction = torch.multinomial(residual, 1, generator=generator).item()
        return {'tokens': accepted_tokens + [correction], 'accepted_count': len(accepted_tokens), 'rejected': True, 'target_positions_checked': i + 1}
    
    # TODO 4：全部接受后，从 target_probs[K] 采样 bonus token
    # 提示：bonus 使用额外的第 K 行 target_probs，而不是最后一个草稿验证位置。
    # bonus = torch.multinomial(???, 1, generator=generator).item()
    return {'tokens': accepted_tokens + [bonus], 'accepted_count': K, 'rejected': False, 'target_positions_checked': K}



In [ ]:
# 按机制拆分测试，分别定位输入契约、接受、residual、bonus 和边界状态。
def _build_speculative_fixture():
    draft_tokens = [0, 1, 2]
    draft_probs = torch.zeros(3, 5)
    target_probs = torch.zeros(4, 5)
    draft_probs[0, 0] = 1.0; target_probs[0, 0] = 1.0
    draft_probs[1, 1] = 1.0; target_probs[1, 3] = 1.0
    draft_probs[2, 2] = 1.0; target_probs[2, 2] = 1.0
    target_probs[3, 4] = 1.0
    return draft_tokens, draft_probs, target_probs

def test_probability_contract():
    draft_tokens, draft_probs, target_probs = _build_speculative_fixture()
    invalid = draft_probs.clone()
    invalid[0, 0] = -0.1
    try:
        speculative_decode_step(invalid, target_probs, draft_tokens)
    except ValueError:
        return
    raise AssertionError('负概率应明确被拒绝')

def test_rejection_and_residual():
    draft_tokens, draft_probs, target_probs = _build_speculative_fixture()
    result = speculative_decode_step(draft_probs, target_probs, draft_tokens)
    assert result['tokens'] == [0, 3]
    assert result['accepted_count'] == 1 and result['rejected']
    assert result['target_positions_checked'] == 2

def test_all_accept_and_bonus():
    _, _, target_probs = _build_speculative_fixture()
    all_accepted = speculative_decode_step(target_probs[:3], target_probs, [0, 3, 2])
    assert all_accepted['tokens'] == [0, 3, 2, 4]
    assert all_accepted['accepted_count'] == 3
    assert not all_accepted['rejected']

def test_zero_q_acceptance():
    zero_q_draft = torch.tensor([[0.0, 1.0, 0.0]])
    zero_q_target = torch.tensor([[0.5, 0.5, 0.0], [0.0, 1.0, 0.0]])
    result = speculative_decode_step(zero_q_draft, zero_q_target, [0])
    assert result['accepted_count'] == 1 and not result['rejected']

def test_zero_residual_failure():
    draft = torch.tensor([[1.0, 0.0]])
    target = torch.tensor([[1.0, 0.0], [1.0, 0.0]])
    try:
        speculative_decode_step(draft, target, [1])
    except ValueError:
        return
    raise AssertionError('residual 概率质量为 0 时应明确报错')

def test_speculative_decoding():
    try:
        torch.manual_seed(42)
        test_probability_contract()
        test_rejection_and_residual()
        test_all_accept_and_bonus()
        test_zero_q_acceptance()
        test_zero_residual_failure()
        print("✅ 投机解码机制测试通过：输入契约、接受、residual 和 bonus 均通过。")
    except NotImplementedError:
        print("请先完成 TODO 代码。")
        raise
    except Exception as error:
        print(f"❌ 投机解码测试失败: {type(error).__name__}: {error}")
        raise

test_speculative_decoding()


## 参考代码与解析

### 代码

In [ ]:
def _validate_probability_contract(draft_probs, target_probs, draft_tokens):
    """检查草稿、目标和候选 token 是否能按位置一一对应。"""
    if draft_probs.dim() != 2 or target_probs.dim() != 2:
        raise ValueError('概率张量必须是二维 [K, vocab]')
    K, vocab_size = draft_probs.shape
    if target_probs.shape != (K + 1, vocab_size) or len(draft_tokens) != K:
        raise ValueError('draft/target/token 的长度或词表维度不匹配')
    if not (torch.isfinite(draft_probs).all() and torch.isfinite(target_probs).all()):
        raise ValueError('概率分布不能包含 NaN 或 Inf')
    # TODO 1：检查概率非负、按行归一化，并确认 token_id 位于词表范围。
    if (draft_probs < 0).any() or (target_probs < 0).any():
        raise ValueError('概率分布不能包含负数')
    ones_draft = torch.ones(K, device=draft_probs.device, dtype=draft_probs.dtype)
    ones_target = torch.ones(K + 1, device=target_probs.device, dtype=target_probs.dtype)
    if not (torch.allclose(draft_probs.sum(-1), ones_draft, atol=1e-5) and torch.allclose(target_probs.sum(-1), ones_target, atol=1e-5)):
        raise ValueError('输入必须是归一化概率分布')
    if any(not isinstance(token_id, int) or not 0 <= token_id < vocab_size for token_id in draft_tokens):
        raise ValueError('draft token id 必须是词表范围内的整数')
    return K, vocab_size

def _accept_probability(target_prob, draft_prob):
    """返回当前候选的接受概率 alpha。"""
    # TODO 2：处理 q=0，并计算 alpha = min(1, p / q)。
    p, q = float(target_prob), float(draft_prob)
    if q == 0.0:
        return 1.0 if p > 0.0 else 0.0
    return min(1.0, p / q)

def _normalized_residual(target_row, draft_row):
    """构造拒绝时用于 correction 的归一化 residual 分布。"""
    # TODO 3：target_row / draft_row 都是 [vocab]；先用 clamp 保留非负差值，再按总质量归一化。
    residual = torch.clamp(target_row - draft_row, min=0)
    residual_mass = residual.sum()
    if residual_mass <= 0:
        raise ValueError('residual 概率质量必须大于 0')
    return residual / residual_mass

def speculative_decode_step(draft_probs, target_probs, draft_tokens, generator=None):
    """完成一次 speculative decoding step。

    `target_probs[:K]` 验证草稿，`target_probs[K]` 生成 bonus token。
    """
    K, vocab_size = _validate_probability_contract(draft_probs, target_probs, draft_tokens)
    accepted_tokens = []
    
    for i in range(K):
        token_id = draft_tokens[i]
        alpha = _accept_probability(target_probs[i, token_id], draft_probs[i, token_id])
        r = torch.rand((), generator=generator).item()
        if r < alpha:
            accepted_tokens.append(int(token_id))
            continue
        residual = _normalized_residual(target_probs[i], draft_probs[i])
        correction = torch.multinomial(residual, 1, generator=generator).item()
        return {'tokens': accepted_tokens + [correction], 'accepted_count': len(accepted_tokens), 'rejected': True, 'target_positions_checked': i + 1}
                
    # TODO 4：全部接受后追加 target 的 bonus token。
    # bonus 来自额外的 target_probs[K]，不是最后一个草稿验证位置。
    bonus = torch.multinomial(target_probs[K], 1, generator=generator).item()
    return {'tokens': accepted_tokens + [bonus], 'accepted_count': K, 'rejected': False, 'target_positions_checked': K}



### 解析

**1. TODO 1：概率输入契约**
- `_validate_probability_contract` 集中检查形状、有限性、非负性、按行归一化和 token 位置；通过后，后续函数只处理自己的概率机制。

**2. TODO 2：接受概率**
- `_accept_probability` 对当前位置读取 $p_i$ 和 $q_i$：$p_i \ge q_i$ 时接受概率为 1，否则为 $\min(1,p_i/q_i)$；$q_i=0$ 时单独处理。

**3. TODO 3：Residual correction**
- `_normalized_residual` 计算 `clamp(target - draft, min=0)` 并归一化；拒绝后从该完整词表分布采样 correction，补回草稿未覆盖的概率质量。

**4. TODO 4：Bonus token 与一次验证**
- 如果 K 个草稿全部接受，还要从 `target_probs[K]` 采样一个 bonus token。
- `target_probs[K]` 是额外的 bonus 位置；它不与任何草稿 token 对齐。

**5. 从机制到基准**
- 本节先确认接受、拒绝、residual correction 与 bonus 的分布语义；35 将它扩展为多 Token 候选的连续前缀验证，68 再在真实模型与 backend 下记录 acceptance rate、目标模型调用、TTFT、TPOT 与吞吐。


## 相关阅读

投机式生成可从经典接受采样、内部草稿和多候选验证三条路径继续阅读；比较实现时应始终同时检查接受率、验证成本与端到端收益。

- [Speculative Sampling 原论文](https://arxiv.org/abs/2302.01318)
- [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- [Medusa 原论文：Multiple Decoding Heads](https://arxiv.org/abs/2401.10782)
- [Transformers Assisted Generation 文档](https://huggingface.co/docs/transformers/main/en/generation_strategies)
- [vLLM Speculative Decoding 文档](https://docs.vllm.ai/en/latest/features/spec_decode.html)
- [35. Multi-Token Decoding | 多 Token 解码](./35_Multi_Token_Decoding.ipynb)
- [68. Speculative Decoding Benchmark | 投机解码基准](./68_Speculative_Decoding_Benchmark.ipynb)